# Experiment A — Calibration of RNA pairing probabilities

*Project 7 · RNA Bioinformatics · Université Paris-Saclay*

## Aim

We examined how well three RNA secondary-structure predictors—ViennaRNA, LinearPartition-V and EternaFold—estimate the probability that a nucleotide is paired. We compared their predicted probabilities with paired/unpaired annotations derived from PDB RNA structures.

The starting reference collection contained 89 structures. We excluded 31 with crossing annotated base pairs because our prediction workflow does not model pseudoknots. This left 58 RNAs covering 3,426 nucleotide positions.

For each position, we summed the predicted probabilities of pairing with all possible partners. We evaluated these nucleotide-level probabilities using expected calibration error (ECE), Brier score, log loss and reliability diagrams. We also used supplementary-section groups to explore uncertainty without treating individual nucleotides as independent samples.

## Input files and running the notebook

This notebook was developed in Google Colab. Run the cells in order. At the upload prompt, provide `Project7_ExperimentA_58_screening_package.zip`, which contains the selected RNA sequences, structural pairing labels and supplementary-section manifest. The reference CT files were used during data preparation; the prediction cells below read the prepared FASTA and CSV files.

The input ZIP is **not yet included in the current GitHub repository**. It must be made available separately, subject to the source data's redistribution terms, before the workflow can be rerun from scratch.

The code below follows our analysis workflow and records the software versions and predictor settings. The generated outputs from the earlier analysis are saved in `Experiment_A/results/58_RNAs/`. We have not yet run this reorganized notebook from beginning to end in a fresh Colab session.

The later 50-RNA and 45-RNA sensitivity analyses, Rfam screening and post-hoc recalibration are documented separately in `Experiment_A/`.


## 1. Python environment

We used ViennaRNA 2.8.0 for partition-function probabilities. LinearPartition-V and EternaFold are compiled from the recorded source-code revisions in the sections below.


In [ ]:
%pip -q install ViennaRNA==2.8.0 pandas numpy matplotlib
import RNA
print("ViennaRNA version:", RNA.__version__)


## 2. Structural-reference data

We load the prepared 58-RNA FASTA file and check the number of sequences before calculating any probabilities. The accompanying CSV files provide the experimental paired/unpaired labels and supplementary-section groups used later in the analysis.


In [ ]:

from google.colab import files
import zipfile
from pathlib import Path

# Load the prepared 58-RNA input package
uploaded = files.upload()

zip_name = next(
    name for name in uploaded
    if name.endswith(".zip")
)

# Unpack FASTA, labels and grouping information
with zipfile.ZipFile(zip_name) as z:
    z.extractall("experiment_a")

# Check the number of FASTA records
fasta = Path("experiment_a/Project7_ExperimentA_58_sequences.fasta")
sequences = fasta.read_text()

rna_count = sequences.count(">")

print("Input package loaded")
print("Number of RNA sequences:", rna_count)


In [ ]:

import RNA
import csv
from pathlib import Path

# Locate the FASTA file from the input package
fasta = next(Path("experiment_a").rglob("*58_sequences.fasta"))

# Read the selected RNA sequences
sequences = {}
name = None

for line in fasta.read_text().splitlines():
    line = line.strip()
    if not line:
        continue
    if line.startswith(">"):
        name = line[1:].split()[0]
        sequences[name] = ""
    else:
        sequences[name] += line.upper()

print("RNA sequences found:", len(sequences))
assert len(sequences) == 58

# Sum base-pair marginals at each nucleotide
results = []

for name, sequence in sequences.items():
    fc = RNA.fold_compound(sequence)
    fc.pf()
    probabilities = fc.bpp(symmetric=True)

    for i, nucleotide in enumerate(sequence, start=1):
        p_paired = sum(
            probabilities[i][j]
            for j in range(1, len(sequence) + 1)
            if j != i
        )

        assert -1e-6 <= p_paired <= 1 + 1e-6

        results.append([
            name, i, nucleotide,
            max(0.0, min(1.0, p_paired))
        ])

    print("Completed:", name)

# Save nucleotide-level predictions
filename = "ExperimentA_ViennaRNA_58_predictions.csv"

with open(filename, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "structure_id",
        "position",
        "nucleotide",
        "predicted_paired_probability"
    ])
    writer.writerows(results)

print("\nViennaRNA predictions completed")
print("RNA structures:", len(sequences))
print("Nucleotide predictions:", len(results))
print("ViennaRNA version:", RNA.__version__)
print("Saved file:", filename)


## 3. ViennaRNA

We used the ViennaRNA partition function to obtain base-pair probabilities. For each nucleotide, we summed these values across potential partners to estimate its probability of being paired. We then matched the predictions to the reference labels using RNA IDs and nucleotide positions.


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

# Read ViennaRNA probabilities
pred = pd.read_csv("ExperimentA_ViennaRNA_58_predictions.csv")

# Read the structural pairing labels
reference_file = next(
    Path("experiment_a").rglob(
        "Project7_ExperimentA_58_reference_labels.csv"
    )
)
ref = pd.read_csv(reference_file)

# Align predictions and reference annotations
keys = ["structure_id", "position"]

assert not pred.duplicated(keys).any()
assert not ref.duplicated(keys).any()

data = pred.merge(
    ref[keys + ["nucleotide", "reference_paired_label"]],
    on=keys,
    how="outer",
    validate="one_to_one",
    indicator=True,
    suffixes=("_pred", "_ref")
)

assert len(data) == 3426
assert (data["_merge"] == "both").all()
assert (data["nucleotide_pred"] == data["nucleotide_ref"]).all()

# Probability and reference-label vectors
p = data["predicted_paired_probability"].to_numpy(float)
y = data["reference_paired_label"].to_numpy(int)

assert np.isfinite(p).all()
assert ((p >= 0) & (p <= 1)).all()
assert set(y).issubset({0, 1})

# Brier score
brier = np.mean((p - y) ** 2)

# Log loss
p_safe = np.clip(p, 1e-12, 1 - 1e-12)
logloss = -np.mean(
    y * np.log(p_safe) +
    (1 - y) * np.log(1 - p_safe)
)

# ECE: 10 equal-width probability bins
bin_ids = np.minimum((p * 10).astype(int), 9)
ece = 0.0
bin_rows = []

for b in range(10):
    mask = bin_ids == b
    if not mask.any():
        continue

    confidence = p[mask].mean()
    observed = y[mask].mean()
    count = mask.sum()

    ece += (count / len(y)) * abs(confidence - observed)

    bin_rows.append({
        "bin": b + 1,
        "count": int(count),
        "mean_predicted": confidence,
        "observed_fraction_paired": observed
    })

# Save matched observations and reliability-bin summaries
data.to_csv("ExperimentA_ViennaRNA_58_matched.csv", index=False)
pd.DataFrame(bin_rows).to_csv(
    "ExperimentA_ViennaRNA_58_calibration_bins.csv",
    index=False
)

print("ViennaRNA calibration metrics")
print("RNAs:", data["structure_id"].nunique())
print("Nucleotides:", len(data))
print("ECE (10 bins):", round(ece, 4))
print("Brier score:", round(brier, 4))
print("Log loss:", round(logloss, 4))


## 4. LinearPartition-V

We ran LinearPartition in thermodynamic mode (`-V`) with a beam size of 100. After checking the output format on one sequence, we calculated pairing probabilities for the full set. We retained the raw probabilities as well as values corrected for minor numerical deviations outside the interval [0, 1].


In [ ]:
import subprocess
from pathlib import Path

# Retrieve LinearPartition source code
if not Path("LinearPartition/.git").exists():
    subprocess.run([
        "git", "clone",
        "https://github.com/LinearFold/LinearPartition.git",
        "LinearPartition"
    ], check=True)

# Use the same source revision as in the analysis
subprocess.run([
    "git", "-C", "LinearPartition", "checkout",
    "b450fb3e63189073b68d385589035f992080aa3a"
], check=True)

# Build LinearPartition
subprocess.run([
    "make", "-C", "LinearPartition",
    "linearpartition", "-j2"
], check=True)

# Confirm that the V-mode executable is available
executable = Path("LinearPartition/bin/linearpartition_v")

assert executable.is_file(), "LinearPartition-V was not built"

commit = subprocess.check_output([
    "git", "-C", "LinearPartition", "rev-parse", "HEAD"
], text=True).strip()

print("LinearPartition-V build completed")
print("Source revision:", commit)

In [ ]:

import subprocess
import sys
from pathlib import Path

# Run LinearPartition through its Python launcher
launcher = Path("LinearPartition/linearpartition").resolve()

# Check output format on one RNA
sequence = sequences["1f1t_A"]

output_folder = Path("LinearPartition_58_outputs")
output_folder.mkdir(exist_ok=True)

output_file = (output_folder / "1f1t_A.bpp").resolve()

run = subprocess.run(
    [
        sys.executable,
        str(launcher),
        "-V",
        "-b", "100",
        "-r", str(output_file)
    ],
    input=sequence + "\n",
    text=True,
    capture_output=True
)

print("Return code:", run.returncode)
print("Program output:", run.stdout)
print("Program messages:", run.stderr)
print("Probability file created:", output_file.exists())

if run.returncode == 0 and output_file.exists():
    lines = output_file.read_text().splitlines()
    print("LinearPartition-V single-sequence check completed")
    print("Number of base-pair probability entries:", len(lines))
    print("First few entries:")
    print("\n".join(lines[:5]))


In [ ]:

import subprocess
import sys
import csv
from pathlib import Path

# Use the same launcher for the full RNA set
launcher = Path("LinearPartition/linearpartition").resolve()
output_folder = Path("LinearPartition_58_outputs")
output_folder.mkdir(exist_ok=True)

assert len(sequences) == 58

results = []
raw_results = []
corrections = 0

for name, sequence in sequences.items():

    bpp_file = (output_folder / f"{name}.bpp").resolve()

    # Clear the previous output before each run
    if bpp_file.exists():
        bpp_file.unlink()

    run = subprocess.run(
        [
            sys.executable,
            str(launcher),
            "-V",
            "-b", "100",
            "-r", str(bpp_file)
        ],
        input=sequence + "\n",
        text=True,
        capture_output=True
    )

    if run.returncode != 0 or not bpp_file.exists():
        raise RuntimeError(
            f"Failed for {name}\n{run.stdout}\n{run.stderr}"
        )

    paired = [0.0] * (len(sequence) + 1)

    for line in bpp_file.read_text().splitlines():
        parts = line.split()

        if not parts:
            continue

        if len(parts) != 3:
            raise ValueError(f"Unexpected line: {line}")

        i, j = int(parts[0]), int(parts[1])
        probability = float(parts[2])

        assert 1 <= i < j <= len(sequence)
        assert 0 <= probability <= 1

        paired[i] += probability
        paired[j] += probability

    for position, nucleotide in enumerate(sequence, 1):

        raw_p = paired[position]

        # Record raw probabilities and bound tiny numerical excursions
        if not -0.001 <= raw_p <= 1.001:
            raise ValueError(
                f"Probability out of range: {name}, "
                f"position {position}: {raw_p}"
            )

        corrected_p = min(1.0, max(0.0, raw_p))

        if corrected_p != raw_p:
            corrections += 1

        raw_results.append([
            name, position, nucleotide, raw_p
        ])

        results.append([
            name, position, nucleotide, corrected_p
        ])

    print("Completed:", name)

columns = [
    "structure_id",
    "position",
    "nucleotide",
    "predicted_paired_probability"
]

for filename, rows in [
    ("ExperimentA_LinearPartitionV_58_predictions.csv", results),
    ("ExperimentA_LinearPartitionV_58_predictions_raw.csv", raw_results)
]:
    with open(filename, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(columns)
        writer.writerows(rows)

print("\nLinearPartition-V predictions completed")
print("RNA structures:", len(sequences))
print("Nucleotide predictions:", len(results))
print("Small numerical corrections:", corrections)
print("Prediction tables saved")


In [ ]:

import pandas as pd
import numpy as np

# Read the ViennaRNA and LinearPartition tables
vienna = pd.read_csv("ExperimentA_ViennaRNA_58_matched.csv")
linear = pd.read_csv("ExperimentA_LinearPartitionV_58_predictions.csv")

keys = ["structure_id", "position"]

# Match observations using identical RNA positions
data = vienna[
    keys + [
        "nucleotide_ref",
        "reference_paired_label",
        "predicted_paired_probability"
    ]
].merge(
    linear[keys + ["nucleotide", "predicted_paired_probability"]],
    on=keys,
    validate="one_to_one",
    how="outer",
    indicator=True,
    suffixes=("_vienna", "_linear")
)

# Check for missing positions or mismatched nucleotides
assert len(data) == 3426
assert (data["_merge"] == "both").all()
assert (data["nucleotide_ref"] == data["nucleotide"]).all()

y = data["reference_paired_label"].to_numpy(int)

def calculate_metrics(p, y):
    p = np.asarray(p, dtype=float)
    assert np.isfinite(p).all()
    assert ((p >= 0) & (p <= 1)).all()

    brier = np.mean((p - y) ** 2)

    safe = np.clip(p, 1e-12, 1 - 1e-12)
    logloss = -np.mean(
        y * np.log(safe) +
        (1 - y) * np.log(1 - safe)
    )

    bin_ids = np.minimum((p * 10).astype(int), 9)
    ece = 0.0
    rows = []

    for b in range(10):
        mask = bin_ids == b
        if mask.any():
            mean_p = p[mask].mean()
            observed = y[mask].mean()
            ece += mask.mean() * abs(mean_p - observed)

            rows.append({
                "bin": b + 1,
                "count": int(mask.sum()),
                "mean_predicted": mean_p,
                "observed_fraction_paired": observed
            })

    return (ece, brier, logloss), pd.DataFrame(rows)

v_metrics, _ = calculate_metrics(
    data["predicted_paired_probability_vienna"], y
)

l_metrics, l_bins = calculate_metrics(
    data["predicted_paired_probability_linear"], y
)

print("ViennaRNA and LinearPartition-V comparison")
print("RNAs:", data["structure_id"].nunique())
print("Nucleotides:", len(data))
print()

print("                  ViennaRNA   LinearPartition-V")
for i, metric in enumerate(["ECE", "Brier score", "Log loss"]):
    print(f"{metric:12s}      {v_metrics[i]:.4f}       {l_metrics[i]:.4f}")

# Save matched observations and reliability-bin data
data.to_csv("ExperimentA_Vienna_vs_Linear_matched.csv", index=False)

l_bins.to_csv(
    "ExperimentA_LinearPartitionV_58_calibration_bins.csv",
    index=False
)

print("\nResults saved.")


## 5. EternaFold

We used `EternaFoldParams.v1` with a posterior probability threshold of `0.00001`. We checked the output format on one RNA before running all 58 sequences. As with the other methods, we obtained nucleotide pairing probabilities by summing the probabilities of each possible pairing partner.


In [ ]:
import subprocess
from pathlib import Path

# Retrieve EternaFold source code
if not Path("EternaFold/.git").exists():
    subprocess.run(
        ["git", "clone",
         "https://github.com/WaymentSteeleLab/EternaFold.git"],
        check=True
    )

# Use the same source revision as in the analysis
subprocess.run([
    "git", "-C", "EternaFold", "checkout",
    "87b9aac55cee14fd562049d08f7b92d3131f10ce"
], check=True)

# Build the EternaFold executable
subprocess.run(
    ["make", "-C", "EternaFold/src", "-j2"],
    check=True
)

# Confirm that the executable and parameters exist
exe = Path("EternaFold/src/contrafold")
params = Path("EternaFold/parameters/EternaFoldParams.v1")

assert exe.is_file(), "EternaFold executable missing"
assert params.is_file(), "EternaFold parameters missing"

revision = subprocess.check_output(
    ["git", "-C", "EternaFold", "rev-parse", "HEAD"],
    text=True
).strip()

print("\nEternaFold build completed")
print("Source revision:", revision)

In [ ]:

import subprocess
from pathlib import Path

# Check the posterior output format on one RNA
sequence = sequences["1f1t_A"]

# Write a sequence input file
input_file = Path("eterna_test.seq").resolve()
input_file.write_text(sequence + "\n")

# Set executable and parameter-file paths
exe = Path("EternaFold/src/contrafold").resolve()
params = Path("EternaFold/parameters/EternaFoldParams.v1").resolve()
output_file = Path("eterna_test_probabilities.txt").resolve()

# Request base-pair posterior probabilities
run = subprocess.run(
    [
        str(exe), "predict", str(input_file),
        "--params", str(params),
        "--posteriors", "0.00001", str(output_file)
    ],
    capture_output=True,
    text=True
)

print("Return code:", run.returncode)
print("Probability file created:", output_file.exists())

if run.returncode != 0:
    print("Error:", run.stderr)
elif output_file.exists():
    lines = output_file.read_text().splitlines()
    print("EternaFold single-sequence check completed")
    print("RNA length:", len(sequence))
    print("Output rows:", len(lines))
    print("First three rows:")
    print("\n".join(lines[:3]))


In [ ]:

import subprocess
import csv
from pathlib import Path

# EternaFold executable and parameter file
exe = Path("EternaFold/src/contrafold").resolve()
params = Path("EternaFold/parameters/EternaFoldParams.v1").resolve()

folder = Path("EternaFold_58_outputs")
folder.mkdir(exist_ok=True)

assert len(sequences) == 58

results = []
raw_results = []
corrections = 0

for name, sequence in sequences.items():

    input_file = (folder / f"{name}.seq").resolve()
    output_file = (folder / f"{name}.bpp").resolve()

    input_file.write_text(sequence + "\n")

    # Remove stale output before rerunning a sequence
    if output_file.exists():
        output_file.unlink()

    run = subprocess.run(
        [
            str(exe), "predict", str(input_file),
            "--params", str(params),
            "--posteriors", "0.00001",
            str(output_file)
        ],
        capture_output=True,
        text=True
    )

    if run.returncode != 0 or not output_file.exists():
        raise RuntimeError(
            f"Failed for {name}\n{run.stdout}\n{run.stderr}"
        )

    paired = [0.0] * (len(sequence) + 1)
    lines = output_file.read_text().splitlines()

    assert len(lines) == len(sequence), name

    for expected_i, line in enumerate(lines, start=1):
        parts = line.split()

        i = int(parts[0])
        nucleotide = parts[1]

        assert i == expected_i
        assert nucleotide == sequence[i - 1]

        for entry in parts[2:]:
            j_text, p_text = entry.split(":")
            j = int(j_text)
            probability = float(p_text)

            assert 1 <= i < j <= len(sequence)
            assert 0 <= probability <= 1

            # Add each reported pair probability to both nucleotides
            paired[i] += probability
            paired[j] += probability

    for position, nucleotide in enumerate(sequence, 1):
        raw_p = paired[position]

        # Correct only small numerical excursions outside [0,1]
        if not -0.001 <= raw_p <= 1.001:
            raise ValueError(
                f"Probability out of range: {name}, "
                f"position {position}: {raw_p}"
            )

        corrected_p = min(1.0, max(0.0, raw_p))

        if corrected_p != raw_p:
            corrections += 1

        raw_results.append([
            name, position, nucleotide, raw_p
        ])

        results.append([
            name, position, nucleotide, corrected_p
        ])

    print("Completed:", name)

columns = [
    "structure_id",
    "position",
    "nucleotide",
    "predicted_paired_probability"
]

for filename, rows in [
    ("ExperimentA_EternaFold_58_predictions.csv", results),
    ("ExperimentA_EternaFold_58_predictions_raw.csv", raw_results)
]:
    with open(filename, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(columns)
        writer.writerows(rows)

print("\nEternaFold predictions completed")
print("RNA structures:", len(sequences))
print("Nucleotide predictions:", len(results))
print("Small numerical corrections:", corrections)
print("Prediction tables saved")


## 6. Calibration results and reliability diagrams

We joined the predictions from all three methods by RNA ID and nucleotide position, ensuring that each predictor was evaluated against the same structural annotations. We calculated ECE with ten equal-width bins, Brier score and log loss, and plotted predicted pairing probability against observed pairing frequency.

The saved results for the 58-RNA set are:

| Predictor | ECE | Brier score | Log loss |
|---|---:|---:|---:|
| ViennaRNA | 0.1107 | 0.1304 | 0.4930 |
| LinearPartition-V | 0.1107 | 0.1304 | 0.4906 |
| EternaFold | 0.0222 | 0.1024 | 0.3321 |

EternaFold had the lowest errors when all nucleotide positions were pooled. We examine sensitivity to RNA grouping and possible relatedness in the accompanying analyses, rather than interpreting this pooled result as evidence of performance on fully independent RNA families.


In [ ]:

import pandas as pd
import numpy as np

# Read the matched ViennaRNA and LinearPartition table
base = pd.read_csv(
    "ExperimentA_Vienna_vs_Linear_matched.csv"
)

# Read EternaFold probabilities
eterna = pd.read_csv(
    "ExperimentA_EternaFold_58_predictions.csv"
)

keys = ["structure_id", "position"]

assert not base.duplicated(keys).any()
assert not eterna.duplicated(keys).any()

# Use a separate probability column for each predictor
base = base.rename(columns={
    "predicted_paired_probability_vienna": "p_vienna",
    "predicted_paired_probability_linear": "p_linear"
})

eterna = eterna.rename(columns={
    "nucleotide": "nucleotide_eterna",
    "predicted_paired_probability": "p_eterna"
})

# Join all predictions on the same reference positions
data = base.merge(
    eterna[keys + ["nucleotide_eterna", "p_eterna"]],
    on=keys,
    how="outer",
    validate="one_to_one",
    indicator="eterna_match"
)

# Verify full positional and nucleotide agreement
assert len(data) == 3426
assert (data["eterna_match"] == "both").all()
assert (data["nucleotide_ref"] == data["nucleotide_eterna"]).all()
assert (data["nucleotide_ref"] == data["nucleotide"]).all()

y = data["reference_paired_label"].to_numpy(int)

# Compare calibration and proper scoring rules
def metrics(p, y):
    p = np.asarray(p, dtype=float)

    assert np.isfinite(p).all()
    assert ((p >= 0) & (p <= 1)).all()

    brier = np.mean((p - y) ** 2)

    safe = np.clip(p, 1e-12, 1 - 1e-12)
    logloss = -np.mean(
        y * np.log(safe)
        + (1 - y) * np.log(1 - safe)
    )

    bin_id = np.minimum((p * 10).astype(int), 9)
    ece = 0.0

    for b in range(10):
        mask = bin_id == b
        if mask.any():
            ece += mask.mean() * abs(
                p[mask].mean() - y[mask].mean()
            )

    return ece, brier, logloss

results = []

for method, column in [
    ("ViennaRNA", "p_vienna"),
    ("LinearPartition-V", "p_linear"),
    ("EternaFold", "p_eterna")
]:
    ece, brier, logloss = metrics(data[column], y)

    results.append({
        "Predictor": method,
        "ECE": ece,
        "Brier": brier,
        "LogLoss": logloss
    })

summary = pd.DataFrame(results)

print("Three-predictor calibration comparison")
print("RNA structures:", data["structure_id"].nunique())
print("Nucleotides:", len(data))
print()
print(summary.round(4).to_string(index=False))

# Save matched predictions and summary metrics
summary.to_csv(
    "ExperimentA_three_predictor_summary.csv",
    index=False
)
data.to_csv(
    "ExperimentA_three_predictor_matched.csv",
    index=False
)

print("\nThree-predictor result tables saved")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

data = pd.read_csv(
    "ExperimentA_three_predictor_matched.csv"
)

y = data["reference_paired_label"].to_numpy(int)

methods = {
    "ViennaRNA": "p_vienna",
    "LinearPartition-V": "p_linear",
    "EternaFold": "p_eterna"
}

plt.figure(figsize=(8, 7))
plt.plot([0, 1], [0, 1], "k--",
         label="Perfect calibration")

for method, column in methods.items():
    p = data[column].to_numpy(float)
    bins = np.minimum((p * 10).astype(int), 9)

    x_values = []
    observed_values = []

    for b in range(10):
        mask = bins == b
        if mask.sum() > 0:
            x_values.append(p[mask].mean())
            observed_values.append(y[mask].mean())

    plt.plot(
        x_values, observed_values,
        "o-", linewidth=2, label=method
    )

plt.xlabel("Mean predicted pairing probability")
plt.ylabel("Observed fraction paired")
plt.title("Experiment A: Three-Predictor Reliability")
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

plt.savefig(
    "ExperimentA_three_predictor_reliability.png",
    dpi=300
)
plt.show()

print("Three-predictor reliability figure saved")

## 7. Grouped bootstrap

Several RNAs share a supplementary-section label. To account for this grouping, we resampled the 35 sections instead of resampling individual nucleotide positions. We used the same sampled sections for all predictors, allowing paired comparisons.

This part of the original 58-RNA workflow uses 2,000 bootstrap replicates and percentile-based 95% intervals. Our later 45-RNA sensitivity analysis uses 5,000 replicates in a separate script. Supplementary-section labels are useful grouping units here, but they are **not verified evolutionary RNA families**.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from google.colab import files

# Read the matched three-predictor table
data = pd.read_csv(
    "ExperimentA_three_predictor_matched.csv"
)

# Read the supplementary-section grouping manifest
manifest_file = next(
    Path("experiment_a").rglob(
        "Project7_ExperimentA_58_manifest.csv"
    )
)

manifest = pd.read_csv(
    manifest_file,
    dtype={"supp_section": str}
)

data = data.merge(
    manifest[["structure_id", "supp_section"]],
    on="structure_id",
    validate="many_to_one"
).reset_index(drop=True)

assert len(data) == 3426
assert data["supp_section"].notna().all()

labels = data["reference_paired_label"].to_numpy(int)

methods = {
    "ViennaRNA": data["p_vienna"].to_numpy(float),
    "LinearPartition-V": data["p_linear"].to_numpy(float),
    "EternaFold": data["p_eterna"].to_numpy(float)
}

def scores(p, y):
    brier = np.mean((p - y) ** 2)

    safe = np.clip(p, 1e-12, 1 - 1e-12)
    logloss = -np.mean(
        y * np.log(safe) +
        (1 - y) * np.log(1 - safe)
    )

    bin_id = np.minimum((p * 10).astype(int), 9)
    ece = 0.0

    for b in range(10):
        mask = bin_id == b
        if mask.any():
            ece += mask.mean() * abs(
                p[mask].mean() - y[mask].mean()
            )

    return np.array([ece, brier, logloss])

# Build the resampling groups from supplementary sections
groups = list(
    data.groupby("supp_section").indices.values()
)

print("RNAs:", data["structure_id"].nunique())
print("Published groups:", len(groups))

# Resample the same sections for every predictor
rng = np.random.default_rng(42)
boot = {name: [] for name in methods}

for repeat in range(2000):
    selected = rng.integers(
        0, len(groups), size=len(groups)
    )

    idx = np.concatenate([groups[i] for i in selected])

    for name, p in methods.items():
        boot[name].append(scores(p[idx], labels[idx]))

for name in methods:
    boot[name] = np.array(boot[name])

metric_names = ["ECE", "Brier", "Log loss"]
rows = []

print("\n95% GROUPED BOOTSTRAP INTERVALS")

for name, p in methods.items():
    observed = scores(p, labels)
    lower, upper = np.percentile(
        boot[name], [2.5, 97.5], axis=0
    )

    print("\n" + name)

    for j, metric in enumerate(metric_names):
        print(
            f"{metric}: {observed[j]:.4f} "
            f"[{lower[j]:.4f}, {upper[j]:.4f}]"
        )

        rows.append({
            "predictor": name,
            "metric": metric,
            "estimate": observed[j],
            "CI_lower": lower[j],
            "CI_upper": upper[j]
        })

# Paired differences use the same bootstrap draws
difference_rows = []

print("\nPAIRED ADVANTAGE OF ETERNAFOLD")
print("Positive numbers favor EternaFold.")

for other in ["ViennaRNA", "LinearPartition-V"]:
    difference = (
        scores(methods[other], labels)
        - scores(methods["EternaFold"], labels)
    )

    boot_difference = boot[other] - boot["EternaFold"]
    lower, upper = np.percentile(
        boot_difference, [2.5, 97.5], axis=0
    )

    print("\nEternaFold versus", other)

    for j, metric in enumerate(metric_names):
        print(
            f"{metric} improvement: {difference[j]:.4f} "
            f"[{lower[j]:.4f}, {upper[j]:.4f}]"
        )

        difference_rows.append({
            "comparison": f"{other} minus EternaFold",
            "metric": metric,
            "improvement": difference[j],
            "CI_lower": lower[j],
            "CI_upper": upper[j]
        })

pd.DataFrame(rows).to_csv(
    "ExperimentA_three_predictor_bootstrap.csv",
    index=False
)

pd.DataFrame(difference_rows).to_csv(
    "ExperimentA_paired_differences.csv",
    index=False
)

print("\nGrouped bootstrap tables saved")

files.download("ExperimentA_three_predictor_bootstrap.csv")
files.download("ExperimentA_paired_differences.csv")

## Interpretation and limitations

EternaFold had lower nucleotide-pooled ECE, Brier score and log loss than ViennaRNA and LinearPartition-V on the 58-RNA reference dataset. The comparison excludes 31 structures with crossing base-pair annotations, so it does not assess pseudoknot-containing structures.

The benchmark's supplementary-section groups are not confirmed evolutionary families. We therefore cannot claim fully family-independent or training-independent evaluation. Separate analyses examine duplicate sequences, possible overlap with released training data, alternative 45-RNA subsets and post-hoc calibration.

**Reproducibility:** We saved the results from our initial analysis. To check that the workflow produces the same results in a new Google Colab session, we need to rerun this notebook with the 58-RNA input files and compare the outputs with the CSV files in `Experiment_A/results/58_RNAs/`. This fresh-run verification has not yet been completed.
